# Inférence de l'âge probable à partir du prénom via Ollama & Gemma3

Ce notebook permet d'utiliser un modèle d'Intelligence Artificielle localisé (`gemma3` via `Ollama`) pour estimer l'âge moyen ou probable associé au prénom du `reviewer_name` dans le fichier des commentaires Airbnb.

In [ ]:
import pandas as pd
import requests
import json
import time
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# Pour afficher la barre de progression tqdm dans pandas
tqdm.pandas()

## 1. Chargement des données

On commence par charger les commentaires du fichier `reviews_select.csv` et à extraire de manière pragmatique les prénoms des évaluateurs.

In [ ]:
input_file = './data/reviews_select.csv'
print(f"Chargement de {input_file}...")
df_reviews = pd.read_csv(input_file, nrows=1000)

# Extraction du prénom (on suppose que le premier mot de 'reviewer_name' équivaut au prénom)
# On s'assure d'abord qu'il s'agit bien d'une chaîne de caractères
df_reviews['first_name'] = df_reviews['reviewer_name'].astype(str).apply(lambda x: x.split()[0].title() if x else '')

print(f"Nombre total de commentaires : {len(df_reviews)}")
df_reviews.head(2)

## 2. Préparation des Prénoms Uniques

Pour ne pas saturer l'API de Ollama et réduire considérablement le temps de calcul, on n'infère l'âge **qu'une seule fois par prénom unique**, puis on fera une fusion (`merge`) avec le dataset complet.

In [ ]:
unique_names_df = pd.DataFrame(df_reviews['first_name'].dropna().unique(), columns=['first_name'])
# Filtrer les prénoms très courts ou manifestement erronés si besoin
unique_names_df = unique_names_df[unique_names_df['first_name'].str.len() > 1]

print(f"{len(unique_names_df)} prénoms uniques ont été identifiés pour l'inférence.")

## 3. Définition de la fonction d'inférence avec Ollama

On prépare une fonction interrogeant l'API locale par défaut de Ollama sur le port `11434` en utilisant le modèle `gemma3`.

In [ ]:
OLLAMA_API_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "gemma3"

def infer_age_from_name_ollama(name, max_retries=2):
    """
    Demande à Gemma3 d'inférer l'âge probable associé à un prénom.
    Le modèle est contraint par un prompt stricte pour ne retourner qu'un nombre.
    """
    prompt = f"Based strictly on cultural assumptions, what is the most typical living age associated with the first name '{name}' today? Answer ONLY with a single numeric value representing the estimated age in years. Do not include any text or punctuation, only the number."
    
    payload = {
        "model": MODEL_NAME,
        "prompt": prompt,
        "stream": False,
        # Optimisations pour la vitesse et la variabilité
        "options": {
            "temperature": 0.1,  # Faible température pour plus de déterminisme
            "num_predict": 10   # Très peu de tokens sont nécessaires
        }
    }
    
    for attempt in range(max_retries):
        try:
            response = requests.post(OLLAMA_API_URL, json=payload, timeout=10)
            if response.status_code == 200:
                result_text = response.json().get("response", "").strip()
                # On filtre la réponse pour extraire uniquement des chiffres, 
                # parfois les LLMs rajoutent des phrases malgré les consignes.
                numeric_digits = ''.join(filter(str.isdigit, result_text))
                if numeric_digits:
                    return int(numeric_digits)
            time.sleep(1)
        except Exception as e:
            time.sleep(1)
            
    return None

### ✨ Test sur quelques prénoms

In [ ]:
test_names = ["Michel", "Kevin", "Liam", "Jean-Pierre", "Emma"]
print(f"Test du modèle {MODEL_NAME}...")
for name in test_names:
    age = infer_age_from_name_ollama(name)
    print(f"{name} : ~{age} ans")
print("Assurez-vous que le service Ollama tourne en local (ollama run gemma3) avant de passer à l'étape suivante.")

## 4. Inférence sur le dataset

*Note: Cette phase peut être longue selon votre configuration hardware (GPU vs CPU) et le nombre de prénoms uniques.*

In [ ]:
print("Démarrage de l'inférence des âges sur la liste des prénoms uniques...")
unique_names_df['inferred_age'] = unique_names_df['first_name'].progress_apply(infer_age_from_name_ollama)

print("Inférence terminée !")
unique_names_df.head(10)

## 5. Fusion et Finalisation

On rattache ces estimations au dataset principal.

In [ ]:
# Fusion par le prénom
df_final = df_reviews.merge(unique_names_df, on='first_name', how='left')

# Traitement des valeurs manquantes si certains prénoms n'ont pas été identifiés par le modèle ou en cas d'erreur API
median_age = df_final['inferred_age'].median()
df_final['inferred_age'].fillna(median_age, inplace=True)
df_final['inferred_age'] = df_final['inferred_age'].round().astype(int)

df_final[['reviewer_name', 'first_name', 'inferred_age']].head(10)

## 6. Analyse visuelle et Sauvegarde

On affiche un graphique de la distribution de l'âge de ceux qui ont commenté.

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df_final['inferred_age'], bins=25, kde=True, color='purple')
plt.title("Distribution de l'âge inféré via Ollama (gemma3)")
plt.xlabel("Âge estimé d'après le prénom")
plt.ylabel("Nombre de reviews correspondantes")
plt.grid(axis='y', alpha=0.5)
plt.show()

In [ ]:
output_file = './data/reviews_select_inferred_age_ollama.csv'
df_final.to_csv(output_file, index=False)
print(f"Le jeu de données comprenant les âges probables a été sauvegardé dans: {output_file}")